In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

In [ ]:
from scipy.stats.distributions import norm

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

### Normal Distribution

Recall the probability density function of the normal distribution:

$$
p(x|\mu, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}} e ^{-(x - \mu)^2/(2\sigma^2)}.
$$

In other words, given some properties of the distribution, the mean $\mu$ and the standard deviation $\sigma$, the function above described the probability *density* as a function of parameter $x\in(-\infty, \infty)$.

So if we want to look at what this distribution looks like a function of $x$, we can just evaluate this function at a bunch of $x$ values and plot it.

In [ ]:
def pdf(x, mu=10, sigma=2.):
    return 1/np.sqrt(2*np.pi*sigma**2) * np.exp(-(x - mu)**2/(2*sigma**2))

In [ ]:
mu = 10.
sigma = 2.

xs = np.linspace(mu - 5*sigma, mu + 5*sigma, 100)

In [ ]:
mu = 10.
sigma = 2.

xs = np.linspace(mu - 5*sigma, mu + 5*sigma, 100)

plt.plot(xs, pdf(xs), ls='--')
plt.ylim(ymin=0)

plt.ylabel(r'$p(x|\mu={}, \sigma={})$'.format(mu, sigma))
plt.xlabel('$x$');

What if we didn't know the function form of the probability density function?  If we had a method of draws samples from the distribution, we can look at the distribution of those samples to get an idea of the shape of the probability density function.

In [ ]:
N = 10000

samples = norm.rvs(loc=mu, scale=sigma, size=N)
plt.hist(samples)

plt.ylabel(r'$\propto p(x|\mu={}, \sigma={})$'.format(mu, sigma))
plt.xlabel('$x$');

In [ ]:
N = 10000

last_samples = samples
samples = norm.rvs(loc=mu, scale=sigma, size=N)
plt.hist(last_samples, histtype='step')
plt.hist(samples, alpha=0.5)

plt.ylabel(r'$\propto p(x|\mu={}, \sigma={})$'.format(mu, sigma))
plt.xlabel('$x$');

The more samples we have, the better we estimate the distribution.

In [ ]:
def plot_dist(N=100, mu=10, sigma=5):
    xs = np.linspace(mu - 5*sigma, mu + 5*sigma, 100)

    plt.plot(xs, norm.pdf(xs, loc=mu, scale=sigma))

    samples = norm.rvs(loc=mu, scale=sigma, size=N)
    plt.hist(samples, bins=30, alpha=0.5, density=True)

    plt.ylabel(r'$p(x|\mu={}, \sigma={})$'.format(mu, sigma))
    plt.xlabel('$x$')
    plt.ylim(0, .3)

In [ ]:
interactive(plot_dist, N=(100, 10000), sigma=(1e-1, 10))

# Monte Carlo Methods

Monte Carlo methods are a class of methods that use randomness to solve problems, typically for the purpose of optimization, numerical integration, and generating samples from a probability distribution.  The name comes from the fact that they were first used to solve problems in the Monte Carlo Casino in Monaco.

In [ ]:
def Pr(w, n, p):
    norm = math.factorial(n) / (math.factorial(w) * math.factorial(n-w))
    return norm * p**w * (1-p)**(n-w)

obs = 'W'
w = obs.count('W')
n = len(obs)

ps = np.linspace(0, 1, 100)
plt.plot(ps, Pr(w, n, p=ps));

# Rejection Sampling

We can draw from $p(x)$ without ever inverting it, provided we have some simpler
distribution $g(x)$ we *can* sample from, scaled so it sits everywhere above the target.

Propose a point from $g$, then keep it with probability $p(x)/Mg(x)$. The points that
survive are distributed according to $p$. The cost is everything we threw away — if $Mg$ is
a poor match to $p$, most proposals are rejected and the method becomes expensive.


Now let's explore ways to generate samples from a distribution when we only know the functional form of its PDF.  Rejection sampling is one approach.

We'll refer to the target distribution that we want to draw samples from as $p(x)$. We can use another distribution $g(x)$ that we *can* draw samples from (whose PDF encompases the whole target distribution after rescaling by some value $k$) to draw samples from the target distribution.

We do so by drawing a sample $z$ from our sampling distribution.  We then generate a random number $u$ unifomly between 0 and $k*g(z)$.  If $u \leq p(z)$ the we save the sample.

In [ ]:
import seaborn as sns

In [ ]:
mu = 10
sigma = 2.

def p(x):
    '''The target distribution to draw samples from'''
    return pdf(x, mu=mu, sigma=sigma)

def g(x):
    '''A distribution we can draw samples from'''
    return np.ones_like(x)


x = np.linspace(-50, 50, 1000)
k = .4 #max(p(x) / g(x))

def rejection_sampling(iter=1000):
    samples = []

    for i in range(iter):
        z = np.random.uniform(-50, 50)
        u = np.random.uniform(0, k*g(z))

        if u <= p(z):
            samples.append(z)

    return np.array(samples)

In [ ]:
plt.plot(x, p(x), label='target distribution ($p(x)$)')
plt.plot(x, k*g(x), label='rescaled sampling Distribution ($g(x)$)')

plt.ylim(0, .7)
plt.legend()
plt.xlabel('$x$');

In [ ]:
s = rejection_sampling(iter=100000)
print("{} samples drawn from target distribution".format(len(s)))

In [ ]:
sns.displot(s, kde=True)
plt.plot(x, p(x))
plt.xlim(0, 20)
plt.xlabel('$x$')
plt.ylabel('$p(x)$');

Now let's sample a slightly more complex distribution.  We'll also switch to a normal sampling distribution.

In [ ]:
def p(x):
    return .5*(pdf(x, mu=mu, sigma=sigma) + \
        pdf(x, mu=mu+5, sigma=sigma/2))

def g(x):
    return pdf(x, mu=mu, sigma=3*sigma)

def g_rvs():
    return norm.rvs(loc=mu, scale=3*sigma)

k = max(p(xs) / g(xs))

In [ ]:
plt.plot(x, p(x), label='target distribution ($p(x)$)')
plt.plot(x, k*g(x), label='rescaled sampling Distribution ($q(x)$)')

plt.ylim(0, .4)
plt.legend(loc='upper left')
plt.xlabel('$x$');

In [ ]:
def rejection_sampling(iter=1000):
    samples = []

    for i in range(iter):
        z = g_rvs()
        u = np.random.uniform(0, k*g(z))

        if u <= p(z):
            samples.append(z)
    return np.array(samples)

In [ ]:
samps = rejection_sampling(iter=100000)

In [ ]:
plt.hist(samps, bins=30, density=True, label='samples')
plt.plot(xs, p(xs), label='target PDF')
plt.legend()
plt.xlim(0, 20)
plt.xlabel('$x$')
plt.ylabel('$p(x)$');

# Importance Sampling

Let's say we are interested in estimating the expectation for some function of our parameter $h(\theta)$ after observing some data $y$, $\mathrm{E}(h(\theta)|y)$, but we can't draw random values of $\theta$ directly from $p(\theta|y)$, but we _can_ draw samples of $\theta$ from some other probability density function $g(\theta)$.

Since we often don't have a normalized posterior density function, we'll use the convention of $q(\theta|y)$ to refer to an unnormalized PDF, where $q(\theta|y) \propto p(\theta|y)$.  In this case:

$$
E(h(\theta)|y) = \int h(\theta) p(\theta|y)d\theta = \frac{\int h(\theta)q(\theta|y) d\theta}{\int q(\theta|y)d\theta}
$$

Putting in some factors of $1$...

$$
E(h(\theta)|y) = \frac{\int \left[ h(\theta)q(\theta|y)/g(\theta)\right] g(\theta)d\theta}{\int \left[q(\theta|y)/g(\theta)\right]g(\theta)d\theta}
$$

Which we can estimate with $S$ draws $\theta^1, \dots, \theta^S$ from $g(\theta)$

$$
E(h(\theta)|y) = \frac{\frac{1}{S}\sum_{s=1}^S h(\theta^s) w(\theta^s)}{\frac{1}{S}\sum_{s=1}^S w(\theta^s)}
$$

where

$$
w(\theta^s) = \frac{q(\theta^s|y)}{g(\theta^s)}
$$

are refered to as the _importance weights_.

Let's estimate the expactation value of $\theta$ (i.e., $h(\theta)=\theta$) from the last example we explored with rejection sampling.

In [ ]:
def q(x):
    '''An unnormalized PDF'''
    return pdf(x, mu=mu, sigma=sigma) + \
        pdf(x, mu=mu+5, sigma=sigma/2)

# Define our sampling distribution g(x) as a random variable X_g
μ_g = mu
σ_g = 3 * sigma
X_g = norm(loc=μ_g, scale=σ_g)

In [ ]:
S = 10000
x_gs = X_g.rvs(S)
ws = q(x_gs)/X_g.pdf(x_gs)

In [ ]:
E_x = np.sum(x_gs * ws)/np.sum(ws)
print(E_x)

Let's compare that to the mean of the samples we drew using rejection sampling.

In [ ]:
np.mean(samps)

In [ ]:
plt.hist(x_gs, bins=50);

In [ ]:
w_max = np.max(ws)
sel = np.random.uniform(0, w_max, S) < ws
plt.hist(x_gs[sel], bins=50, density=True);
plt.plot(xs, p(xs), label='target PDF')
plt.legend()
plt.xlim(0, 20)
plt.xlabel('$x$')
plt.ylabel('$p(x)$');

# Markov Chain Monte Carlo (MCMC)

Rejection and importance sampling both need a proposal distribution that covers the target
reasonably well. In more than a few dimensions that becomes hopeless: the region where $p$ is
large is a vanishing fraction of the space, and almost every proposal is wasted.

MCMC takes a different approach. Instead of drawing independent samples, we build a
**Markov chain** — a sequence where each point is generated from the one before it, and from
nothing else. Arrange the transition rule correctly and the chain spends time in each region
in proportion to $p$, so the sequence it traces *is* a sample from the target.

The price is that consecutive points are **correlated**, because each one was generated from
its predecessor. A chain of 10,000 points is worth rather fewer than 10,000 independent
draws, and part of running an MCMC properly is knowing how many fewer.


# Metropolis algorithm

Here we're going to build a simple Metroplis sampler, and use it to draw samples from a target distribution.

The algorithm works in the following way:
1. The chain is at location $x$
1. A new location $x'$ is drawn from a proposal distribution $q(x)$, which we'll use $q(x)\sim \mathcal{N}(x, \sigma)$.
1. The ratio of the target probability density at the proposed location to the current location is calculated, $\alpha = \frac{p(x')}{p(x)}$.
1. If $\alpha>1$ the jump is accepted, if $\alpha<1$ it's accepted with a probability of $\alpha$.  If a jump is rejected the current sample $x$ is repeated in the chain.

In [ ]:
x0 = -7.

x = x0
p_current = p(x0)

chain = [x]
probs = [p_current]

In [ ]:
niter = 10000
sigma_jump = 5.

for i in range(niter):
    xp = norm.rvs(loc=x, scale=sigma_jump)
    p_p = p(xp)
    
    α = p_p/p_current
    u = np.random.uniform()
    accepted = u < α
    
    if accepted:
        x = xp
        p_current = p_p
    chain.append(x)
    probs.append(p_current)

In [ ]:
burnin_length = 2000

plt.plot(chain[burnin_length:])
plt.xlabel('iteration')
plt.ylabel('$x$');

In [ ]:
burnin_length = 2000

plt.plot(chain[burnin_length:],'.')
plt.xlabel('iteration')
plt.ylabel('$x$');

What's left looks like it's fairly uncorrelated, which is what we're looking for.  There are still correlations in the chain that, strictly speaking, should be dealt with (by thinning out the chain), but we'll call this good enough for our purprose, and take a look at the histogram, which gives us an idea of target distribution.

In [ ]:
plt.hist(chain, density=True, bins=30, label='chain')
plt.plot(xs, p(xs), color='r', label='target PDF')
plt.xlabel('$x$')
plt.ylabel('$p(x)$');

---
## How correlated is the chain, really?

Above we said the chain looked "fairly uncorrelated" and moved on. That was a guess made by
eye. It is also the single most common way to fool yourself with MCMC, so let us measure it
instead.

Consecutive samples are correlated by construction. The **autocorrelation function** tells us
how far apart two points have to be before that memory has faded:

$$\rho(k) = \frac{\langle (x_i - \bar{x})(x_{i+k} - \bar{x}) \rangle}{\langle (x_i - \bar{x})^2 \rangle}$$


In [ ]:
def autocorr(chain, maxlag=200):
    """Autocorrelation of a chain at lags 0..maxlag-1."""
    c = np.asarray(chain, dtype=float)
    c = c - c.mean()
    denom = c @ c
    return np.array([1.0 if k == 0 else (c[:-k] @ c[k:]) / denom
                     for k in range(maxlag)])


samples = np.array(chain[burnin_length:])
rho = autocorr(samples)

plt.plot(rho)
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('lag $k$')
plt.ylabel(r'$\rho(k)$')
plt.xlim(0, 40);


The correlation dies away after a handful of steps. The scale on which it does so is the
**autocorrelation time** $\tau$:

$$\tau = 1 + 2\sum_{k=1}^{\infty} \rho(k)$$

In practice the sum is truncated where $\rho$ first goes negative — beyond that point we are
summing noise.

$\tau$ has a direct interpretation: **roughly every $\tau$-th sample is independent.** So a
chain of $N$ points carries the information of

$$N_\mathrm{eff} = \frac{N}{\tau}$$

independent draws. This is the **effective sample size**, and it is the number that should
appear in any uncertainty you quote — not the length of the chain.


In [ ]:
def effective_sample_size(chain, maxlag=200):
    """Effective sample size, and the autocorrelation time it comes from."""
    rho = autocorr(chain, maxlag)
    negative = np.where(rho < 0)[0]
    stop = negative[0] if len(negative) else len(rho)
    tau = 1 + 2 * rho[1:stop].sum()
    return len(chain) / tau, tau


n_eff, tau = effective_sample_size(samples)
print(f'chain length      {len(samples)}')
print(f'autocorr time tau {tau:.1f}')
print(f'effective samples {n_eff:.0f}   ({n_eff/len(samples):.1%} of the chain)')


### The step size controls all of this

`sigma_jump` sets how far the sampler tries to move each iteration, and it trades off against
itself:

- **Too small** — nearly every proposal is accepted, but the chain inches along and takes
  forever to cross the distribution.
- **Too large** — proposals land far out in the tails where $p$ is tiny, almost everything is
  rejected, and the chain sits still.

Both give long autocorrelation times. Let us measure all three cases.


In [ ]:
def run_chain(sigma_jump, niter=10000, x0=-7., seed=0):
    rng = np.random.default_rng(seed)
    x, p_current = x0, p(x0)
    chain = [x]
    for _ in range(niter):
        xp = x + rng.normal(scale=sigma_jump)
        p_p = p(xp)
        if rng.uniform() < p_p / p_current:
            x, p_current = xp, p_p
        chain.append(x)
    return np.array(chain)


print(f'{"sigma_jump":>11}  {"accept":>7}  {"tau":>7}  {"N_eff":>7}  {"efficiency":>10}')
for sj in (0.2, 5.0, 50.0):
    c = run_chain(sj)[burnin_length:]
    accept = np.mean(np.diff(c) != 0)
    n_eff, tau = effective_sample_size(c)
    print(f'{sj:11.1f}  {accept:7.2f}  {tau:7.1f}  {n_eff:7.0f}  {n_eff/len(c):10.1%}')


**Look at the first row.** A 95% acceptance rate sounds like a sampler doing well — almost
every proposal accepted, nothing wasted. It yields about **25 times fewer** effective samples
than the middle case: 54 useful draws out of 8000, against 1329.

A high acceptance rate is not a sign of success. It usually means the sampler is barely
moving, and the trace plot will look reassuringly smooth while telling you almost nothing.

This matters directly for your homework: when you report a posterior mean and its uncertainty
from a chain, the uncertainty goes as $1/\sqrt{N_\mathrm{eff}}$ — **not** $1/\sqrt{N}$.
Quoting the chain length instead can understate your error bar by a large factor.

*(For a target this simple you can tune by hand. Rules of thumb exist — acceptance rates
around 0.2–0.5 are often quoted for random-walk Metropolis — but measuring $N_\mathrm{eff}$
is the honest check.)*
